# OASM-10 gridded prediction

Version 6 uses the same feature definitions and fitted preprocessing as station evaluation. Inputs remain in the pre-standardization scale; Landsat normalized differences are DN-derived. The slope transform uses epsilon 0.001. SoilGrids and precipitation gaps use saved training-only imputation values; no statistics are fitted on the requested map.

Each deeper prediction is paired with its own outer fold's 5 cm predictor. The five paired outputs are averaged at the end. The stored source raster affine transform is retained to avoid a half-pixel shift. Output grid spacing is nominal 10 m in EPSG:4326, not a claim of independently validated 10 m information at depth.

Authenticate Google Earth Engine before online extraction. For an offline example, load an archived feature grid instead. Optical selection is retrospective within +/-14 days, so this is not an instantaneous nowcast.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "feature_schema.json").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Run this notebook inside the OASM-10 repository folder.")
    ROOT = ROOT.parent
REV = ROOT
sys.path.insert(0, str(REV))
import torch
torch.set_num_threads(8)
torch.set_float32_matmul_precision("high")


In [ ]:
DEPTHS = [5, 20, 50]
TARGET_TIME = "2023-10-18T00:00:00"
# [west, south, east, north]; a small agricultural example.
BOUNDS = [-93.82, 42.00, -93.80, 42.02]
OUTPUT = ROOT / "outputs" / TARGET_TIME[:10]
OUTPUT.mkdir(parents=True, exist_ok=True)
# Use ROOT / "outputs/retrained/models" after retraining.
MODEL_ROOT = ROOT / "models"
CLIMATE_RASTER = ROOT / "data/ancillary/Beck_KG_V1_present_0p0083.tif"
# Set your own Earth Engine Cloud project, or use an already configured project.
EE_PROJECT = None


In [ ]:
from oasm.gee import fetch_feature_grid
# Authenticate first with `earthengine authenticate` (see README.md).
frame, grid = fetch_feature_grid(BOUNDS, TARGET_TIME, OUTPUT / "features",
                                 climate_raster=CLIMATE_RASTER, project=EE_PROJECT)


In [ ]:
from oasm.inference import predict_product, write_geotiff
from affine import Affine
import numpy as np
prediction = predict_product(frame, MODEL_ROOT, depths=DEPTHS)
transform = Affine(*grid["transform"])
for depth in DEPTHS:
    raster = prediction[f"sm_{depth}cm"].to_numpy().reshape(grid["height"], grid["width"])
    path = OUTPUT / f"OASM10_{depth}cm_{TARGET_TIME[:10]}.tif"
    write_geotiff(path, raster, transform, tags={"model_version":"oasm10_public_release", "target_time":TARGET_TIME, "depth_cm":str(depth), "units":"m3 m-3", "grid_definition":"nominal 10 m EPSG:4326", "postprocessing":"unclipped estimates; out-of-range flags supplied separately"})
prediction.to_parquet(OUTPUT / "prediction_quality_flags.parquet", index=False)
print(OUTPUT)
